# Data Extraction: UCI Online Retail II

Loads and cleans the UCI Online Retail II dataset, computes customer-level features for CLV prediction.

- **Source:** [UCI ML Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii) --- real UK e-commerce transactions
- **Output:** `data/raw/clv_data.csv`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Raw Data

In [2]:
# Load both sheets from the Excel file and concatenate
sheet1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
sheet2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df_raw = pd.concat([sheet1, sheet2], ignore_index=True)

print(f"Sheet 1 (2009-2010): {len(sheet1):,} rows")
print(f"Sheet 2 (2010-2011): {len(sheet2):,} rows")
print(f"Combined: {len(df_raw):,} rows")
print(f"\n=== Dtypes ===")
print(df_raw.dtypes)
print(f"\n=== Missing Values ===")
print(df_raw.isna().sum())
print(f"\n=== Sample ===")
df_raw.head()

Sheet 1 (2009-2010): 525,461 rows
Sheet 2 (2010-2011): 541,910 rows
Combined: 1,067,371 rows

=== Dtypes ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

=== Missing Values ===
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

=== Sample ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Data Cleaning

In [3]:
rows_before = len(df_raw)
df = df_raw.copy()

# --- Step 1: Convert Invoice to string for cancellation detection ---
df['Invoice'] = df['Invoice'].astype(str).str.strip()

# --- Step 2: Flag cancellations (Invoice starts with 'C') ---
df['is_cancellation'] = df['Invoice'].str.startswith('C')
n_cancellations = df['is_cancellation'].sum()
print(f"Cancellations detected: {n_cancellations:,} rows ({n_cancellations / len(df):.1%})")

# --- Step 3: Save cancellation counts per customer BEFORE removing them ---
# Only count cancellations for rows that have a Customer ID
cancel_mask = df['is_cancellation'] & df['Customer ID'].notna()
cancellation_counts = (
    df[cancel_mask]
    .groupby('Customer ID')['Invoice']
    .nunique()
    .rename('cancellation_orders')
)
print(f"Customers with cancellations: {len(cancellation_counts):,}")

# Also count total orders (incl. cancellations) per customer for rate calculation
total_orders_with_cancellations = (
    df[df['Customer ID'].notna()]
    .groupby('Customer ID')['Invoice']
    .nunique()
    .rename('total_orders_incl_cancellations')
)

# --- Step 4: Drop rows with null Customer ID ---
n_null_cust = df['Customer ID'].isna().sum()
df = df[df['Customer ID'].notna()].copy()
df['Customer ID'] = df['Customer ID'].astype(int)
print(f"Dropped null Customer ID: {n_null_cust:,} rows ({n_null_cust / rows_before:.1%})")

# --- Step 5: Remove cancellations ---
n_before_cancel = len(df)
df = df[~df['is_cancellation']].copy()
print(f"Removed cancellations: {n_before_cancel - len(df):,} rows")

# --- Step 6: Remove rows with Quantity <= 0 or Price <= 0 ---
n_before_qty = len(df)
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)].copy()
print(f"Removed non-positive Quantity/Price: {n_before_qty - len(df):,} rows")

# --- Step 7: Remove non-product StockCodes ---
# Filter out StockCodes that are purely alphabetic and short (service codes, fees, etc.)
non_product_codes = {'POST', 'DOT', 'M', 'BANK CHARGES', 'PADS', 'S', 'CRUK',
                     'AMAZONFEE', 'D', 'C2', 'DCGS', 'DCGSSBOY', 'DCGSSGIRL',
                     'SP1002', 'gift_0001_10', 'gift_0001_20', 'gift_0001_30',
                     'gift_0001_40', 'gift_0001_50'}
# Also remove any StockCode that is purely alphabetic and <= 5 chars (catches misc. service codes)
is_alpha_short = df['StockCode'].astype(str).str.match(r'^[A-Za-z]{1,5}$')
is_known_non_product = df['StockCode'].astype(str).str.upper().isin(
    {code.upper() for code in non_product_codes}
)
non_product_mask = is_alpha_short | is_known_non_product
n_non_product = non_product_mask.sum()
df = df[~non_product_mask].copy()
print(f"Removed non-product StockCodes: {n_non_product:,} rows")

# --- Step 8: Deduplicate ---
n_before_dedup = len(df)
df = df.drop_duplicates().copy()
print(f"Removed duplicates: {n_before_dedup - len(df):,} rows")

# --- Step 9: Compute line_total ---
df['line_total'] = df['Quantity'] * df['Price']

# --- Drop helper column ---
df = df.drop(columns=['is_cancellation'])

# --- Summary ---
print(f"\n{'='*50}")
print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning:  {len(df):,} ({len(df)/rows_before:.1%} retained)")
print(f"Unique customers:     {df['Customer ID'].nunique():,}")
print(f"Unique invoices:      {df['Invoice'].nunique():,}")
print(f"Date range:           {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Cancellations detected: 19,494 rows (1.8%)
Customers with cancellations: 2,572
Dropped null Customer ID: 243,007 rows (22.8%)
Removed cancellations: 18,744 rows
Removed non-positive Quantity/Price: 71 rows


Removed non-product StockCodes: 2,872 rows
Removed duplicates: 26,055 rows

Rows before cleaning: 1,067,371
Rows after cleaning:  776,622 (72.8% retained)
Unique customers:     5,861
Unique invoices:      36,639
Date range:           2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 3. Temporal Split

In [4]:
# Temporal boundaries: ~6 months before end of dataset
calibration_end = pd.Timestamp('2011-06-09')
cutoff_date = pd.Timestamp('2011-12-09')  # end of dataset

calibration = df[df['InvoiceDate'] < calibration_end].copy()
holdout = df[(df['InvoiceDate'] >= calibration_end) & (df['InvoiceDate'] < cutoff_date)].copy()

cal_customers = set(calibration['Customer ID'].unique())
hold_customers = set(holdout['Customer ID'].unique())
overlap = cal_customers & hold_customers

print(f"=== Temporal Split ===")
print(f"Calibration period:   {calibration['InvoiceDate'].min().date()} to {calibration_end.date()}")
print(f"Holdout period:       {calibration_end.date()} to {cutoff_date.date()}")
print(f"")
print(f"Calibration rows:     {len(calibration):,}")
print(f"Holdout rows:         {len(holdout):,}")
print(f"")
print(f"Calibration customers: {len(cal_customers):,}")
print(f"Holdout customers:     {len(hold_customers):,}")
print(f"Overlap (in both):     {len(overlap):,} ({len(overlap)/len(cal_customers):.1%} of calibration)")
print(f"Holdout-only (new):    {len(hold_customers - cal_customers):,}")

=== Temporal Split ===
Calibration period:   2009-12-01 to 2011-06-09
Holdout period:       2011-06-09 to 2011-12-09

Calibration rows:     536,391
Holdout rows:         239,626

Calibration customers: 4,951
Holdout customers:     3,485
Overlap (in both):     2,576 (52.0% of calibration)
Holdout-only (new):    909


## 4. Order-Level Aggregation

In [5]:
# Aggregate line items to order level for calibration period
# One row per (Customer ID, Invoice)
cal_orders = calibration.groupby(['Customer ID', 'Invoice']).agg(
    order_date=('InvoiceDate', 'min'),
    order_revenue=('line_total', 'sum'),
    n_items=('StockCode', 'nunique')  # basket diversity: distinct products per order
).reset_index()

print(f"Calibration orders: {len(cal_orders):,}")
print(f"Unique customers:   {cal_orders['Customer ID'].nunique():,}")
print(f"\nOrders per customer:")
print(cal_orders.groupby('Customer ID').size().describe().round(2))
print(f"\nOrder revenue:")
print(cal_orders['order_revenue'].describe().round(2))

Calibration orders: 25,996
Unique customers:   4,951

Orders per customer:
count    4951.00
mean        5.25
std        10.03
min         1.00
25%         1.00
50%         3.00
75%         6.00
max       240.00
dtype: float64

Order revenue:
count    25996.00
mean       452.58
std       1020.00
min          0.55
25%        157.75
50%        301.50
75%        470.17
max      77183.60
Name: order_revenue, dtype: float64


## 5. Customer-Level Feature Engineering

In [6]:
# --- RFM Core Features ---
customer_agg = cal_orders.groupby('Customer ID').agg(
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max'),
    total_orders=('Invoice', 'nunique'),
    total_spend=('order_revenue', 'sum'),
    avg_order_value=('order_revenue', 'mean'),
    avg_basket_size=('n_items', 'mean'),
).reset_index()

# frequency = distinct orders - 1 (repeat purchases)
customer_agg['frequency'] = customer_agg['total_orders'] - 1

# recency = days from first to last order
customer_agg['recency'] = (
    customer_agg['last_order_date'] - customer_agg['first_order_date']
).dt.days

# T = customer age in days at calibration end
customer_agg['T'] = (
    calibration_end - customer_agg['first_order_date']
).dt.days

# days_since_last_order = days from last order to calibration end
customer_agg['days_since_last_order'] = (
    calibration_end - customer_agg['last_order_date']
).dt.days

# days_active = days from first to last order
customer_agg['days_active'] = (
    customer_agg['last_order_date'] - customer_agg['first_order_date']
).dt.days

# --- monetary_value: avg revenue on REPEAT transactions only ---
# For customers with frequency > 0, compute mean of all order revenues
# excluding the first order (BG/NBD convention: monetary_value = avg of repeat txns)
# For one-time buyers (frequency = 0), fall back to avg_order_value

def compute_monetary_value(group):
    """Avg order revenue on repeat transactions (all but first order).
    Falls back to avg_order_value for one-time buyers."""
    sorted_orders = group.sort_values('order_date')
    if len(sorted_orders) > 1:
        return sorted_orders.iloc[1:]['order_revenue'].mean()
    else:
        return sorted_orders['order_revenue'].mean()

monetary_values = cal_orders.groupby('Customer ID').apply(compute_monetary_value)
monetary_values.name = 'monetary_value'
customer_agg = customer_agg.merge(monetary_values, on='Customer ID', how='left')

# --- Unique products across all calibration orders ---
unique_products = (
    calibration.groupby('Customer ID')['StockCode']
    .nunique()
    .rename('unique_products')
    .reset_index()
)
customer_agg = customer_agg.merge(unique_products, on='Customer ID', how='left')

# --- Purchase regularity: std of inter-purchase intervals ---
def compute_purchase_regularity(group):
    """Std of days between consecutive orders.
    For customers with only 1 order, return NaN (filled with 999 below)."""
    dates = group.sort_values('order_date')['order_date']
    if len(dates) < 2:
        return np.nan
    intervals = dates.diff().dropna().dt.days
    if len(intervals) < 2:
        # Only 2 orders = 1 interval, std is 0
        return 0.0
    return intervals.std()

purchase_regularity = (
    cal_orders.groupby('Customer ID')
    .apply(compute_purchase_regularity)
    .rename('purchase_regularity')
    .reset_index()
)
customer_agg = customer_agg.merge(purchase_regularity, on='Customer ID', how='left')
# Fill NaN (one-time buyers) with 999 = highly irregular
customer_agg['purchase_regularity'] = customer_agg['purchase_regularity'].fillna(999)

# --- Cancellation rate (from pre-cleaning data) ---
customer_agg = customer_agg.merge(
    cancellation_counts.reset_index().rename(columns={'Customer ID': 'Customer ID'}),
    on='Customer ID', how='left'
)
customer_agg['cancellation_orders'] = customer_agg['cancellation_orders'].fillna(0).astype(int)

customer_agg = customer_agg.merge(
    total_orders_with_cancellations.reset_index().rename(columns={'Customer ID': 'Customer ID'}),
    on='Customer ID', how='left'
)
customer_agg['cancellation_rate'] = (
    customer_agg['cancellation_orders'] / customer_agg['total_orders_incl_cancellations']
).fillna(0)

# --- Country: most frequent per customer ---
country = (
    calibration.groupby('Customer ID')['Country']
    .agg(lambda x: x.value_counts().index[0])
    .rename('country')
    .reset_index()
)
customer_agg = customer_agg.merge(country, on='Customer ID', how='left')

# --- Clean up: drop intermediate date columns ---
customer_agg = customer_agg.drop(
    columns=['first_order_date', 'last_order_date',
             'cancellation_orders', 'total_orders_incl_cancellations']
)

print(f"Customer features computed: {len(customer_agg):,} customers")
print(f"Feature columns: {list(customer_agg.columns)}")
print(f"\n=== Feature Dtypes ===")
print(customer_agg.dtypes)

Customer features computed: 4,951 customers
Feature columns: ['Customer ID', 'total_orders', 'total_spend', 'avg_order_value', 'avg_basket_size', 'frequency', 'recency', 'T', 'days_since_last_order', 'days_active', 'monetary_value', 'unique_products', 'purchase_regularity', 'cancellation_rate', 'country']

=== Feature Dtypes ===
Customer ID                int64
total_orders               int64
total_spend              float64
avg_order_value          float64
avg_basket_size          float64
frequency                  int64
recency                    int64
T                          int64
days_since_last_order      int64
days_active                int64
monetary_value           float64
unique_products            int64
purchase_regularity      float64
cancellation_rate        float64
country                   object
dtype: object


## 6. Holdout Labels

In [7]:
# Compute holdout labels: transactions and revenue in holdout period
holdout_labels = holdout.groupby('Customer ID').agg(
    actual_holdout_transactions=('Invoice', 'nunique'),
    actual_holdout_revenue=('line_total', 'sum')
).reset_index()

print(f"Customers with holdout activity: {len(holdout_labels):,}")
print(f"Avg holdout transactions: {holdout_labels['actual_holdout_transactions'].mean():.2f}")
print(f"Avg holdout revenue: {holdout_labels['actual_holdout_revenue'].mean():.2f}")

# LEFT JOIN to customer features (customers not in holdout get 0)
customers = customer_agg.merge(holdout_labels, on='Customer ID', how='left')
customers['actual_holdout_transactions'] = customers['actual_holdout_transactions'].fillna(0).astype(int)
customers['actual_holdout_revenue'] = customers['actual_holdout_revenue'].fillna(0.0)

print(f"\nAfter LEFT JOIN: {len(customers):,} customers")
print(f"With holdout activity: {(customers['actual_holdout_transactions'] > 0).sum():,}")
print(f"Without holdout activity: {(customers['actual_holdout_transactions'] == 0).sum():,}")

Customers with holdout activity: 3,485
Avg holdout transactions: 3.04
Avg holdout revenue: 1470.17

After LEFT JOIN: 4,951 customers
With holdout activity: 2,576
Without holdout activity: 2,375


## 7. Final Assembly and Filtering

In [8]:
# Binary holdout label
customers['purchased_in_holdout'] = (customers['actual_holdout_transactions'] > 0).astype(int)

# Filter: T > 7 (exclude customers with < 1 week of history)
n_before_t = len(customers)
customers = customers[customers['T'] > 7].copy()
print(f"Filtered T > 7: {n_before_t:,} -> {len(customers):,} ({n_before_t - len(customers):,} removed)")

# Ensure monetary_value > 0 (required for Gamma-Gamma model)
n_before_mv = len(customers)
customers = customers[customers['monetary_value'] > 0].copy()
print(f"Filtered monetary_value > 0: {n_before_mv:,} -> {len(customers):,} ({n_before_mv - len(customers):,} removed)")

# Rename Customer ID to user_id for consistency with downstream notebooks
customers = customers.rename(columns={'Customer ID': 'user_id'})

# Reorder columns for readability
column_order = [
    # ID
    'user_id',
    # RFM core
    'frequency', 'recency', 'T', 'monetary_value',
    # Transaction context
    'total_orders', 'total_spend', 'avg_order_value', 'days_since_last_order',
    # UCI-specific features
    'unique_products', 'avg_basket_size', 'purchase_regularity',
    'days_active', 'cancellation_rate',
    # Geography
    'country',
    # Holdout labels
    'actual_holdout_transactions', 'actual_holdout_revenue', 'purchased_in_holdout',
]
customers = customers[column_order].reset_index(drop=True)

# --- Summary ---
positive_rate = customers['purchased_in_holdout'].mean()
one_time_rate = (customers['frequency'] == 0).mean()

print(f"\n{'='*50}")
print(f"Final customer count:     {len(customers):,}")
print(f"Holdout positive rate:    {positive_rate:.1%} ({customers['purchased_in_holdout'].sum():,} customers)")
print(f"One-time buyer rate:      {one_time_rate:.1%} ({(customers['frequency'] == 0).sum():,} customers)")
print(f"Repeat buyer rate:        {1 - one_time_rate:.1%} ({(customers['frequency'] > 0).sum():,} customers)")

Filtered T > 7: 4,951 -> 4,918 (33 removed)
Filtered monetary_value > 0: 4,918 -> 4,918 (0 removed)

Final customer count:     4,918
Holdout positive rate:    52.0% (2,557 customers)
One-time buyer rate:      31.1% (1,528 customers)
Repeat buyer rate:        68.9% (3,390 customers)


## 8. Summary Statistics

In [9]:
print(f"=== Dataset Overview ===")
print(f"Total customers:          {len(customers):,}")
print(f"Holdout positive rate:    {customers['purchased_in_holdout'].mean():.1%}")
print(f"One-time buyer %:         {(customers['frequency'] == 0).mean():.1%}")
print(f"")

# Feature distributions
print(f"=== Feature Distributions ===")
numeric_cols = customers.select_dtypes(include='number').columns.drop('user_id')
print(customers[numeric_cols].describe().round(2).to_string())

# Frequency distribution
print(f"\n=== Frequency Distribution ===")
freq_dist = customers['frequency'].value_counts().sort_index()
for freq_val, count in freq_dist.head(10).items():
    print(f"  frequency={freq_val}: {count:,} customers ({count/len(customers):.1%})")
if len(freq_dist) > 10:
    remaining = freq_dist.iloc[10:].sum()
    print(f"  frequency>={freq_dist.index[10]}: {remaining:,} customers ({remaining/len(customers):.1%})")

# Country distribution (top 10)
print(f"\n=== Country Distribution (Top 10) ===")
country_dist = customers['country'].value_counts()
for country_name, count in country_dist.head(10).items():
    print(f"  {country_name}: {count:,} ({count/len(customers):.1%})")
other_count = country_dist.iloc[10:].sum() if len(country_dist) > 10 else 0
if other_count > 0:
    print(f"  Other ({len(country_dist) - 10} countries): {other_count:,} ({other_count/len(customers):.1%})")

# Holdout revenue distribution for purchasers
purchasers = customers[customers['purchased_in_holdout'] == 1]
print(f"\n=== Holdout Revenue (purchasers only, n={len(purchasers):,}) ===")
print(purchasers['actual_holdout_revenue'].describe().round(2).to_string())

=== Dataset Overview ===
Total customers:          4,918
Holdout positive rate:    52.0%
One-time buyer %:         31.1%

=== Feature Distributions ===
       frequency  recency        T  monetary_value  total_orders  total_spend  avg_order_value  days_since_last_order  unique_products  avg_basket_size  purchase_regularity  days_active  cancellation_rate  actual_holdout_transactions  actual_holdout_revenue  purchased_in_holdout
count    4918.00  4918.00  4918.00         4918.00       4918.00      4918.00          4918.00                4918.00          4918.00          4918.00              4918.00      4918.00            4918.00                      4918.00                 4918.00               4918.00
mean        4.28   192.68   365.01          356.26          5.28      2389.98           367.31                 171.98            70.89            20.91               337.99       192.68               0.12                         1.76                  887.35                  0.52
std     

## 9. Save Output

In [10]:
os.makedirs('../data/raw', exist_ok=True)
customers.to_csv('../data/raw/clv_data.csv', index=False)

print(f"Saved to data/raw/clv_data.csv")
print(f"  Shape: {customers.shape[0]:,} customers x {customers.shape[1]} features")
print(f"  Columns: {list(customers.columns)}")
print(f"\nNext: 02_purchase_propensity.ipynb")

Saved to data/raw/clv_data.csv
  Shape: 4,918 customers x 18 features
  Columns: ['user_id', 'frequency', 'recency', 'T', 'monetary_value', 'total_orders', 'total_spend', 'avg_order_value', 'days_since_last_order', 'unique_products', 'avg_basket_size', 'purchase_regularity', 'days_active', 'cancellation_rate', 'country', 'actual_holdout_transactions', 'actual_holdout_revenue', 'purchased_in_holdout']

Next: 02_purchase_propensity.ipynb
